In [2]:
pip install gurobipy

   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.1 MB 4.2 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/11.1 MB 3.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.4/11.1 MB 4.5 MB/s eta 0:00:02
   ------------ --------------------------- 3.4/11.1 MB 4.5 MB/s eta 0:00:02
   ------------- -------------------------- 3.7/11.1 MB 4.4 MB/s eta 0:00:02
   --------------- ------------------------ 4.2/11.1 MB 4.1 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/11.1 MB 3.3 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.1 MB 3.6 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.1 MB 3.8 MB/s eta 0:00:02
   ------------------------- -------------- 7.1/11.1 MB 3.5 MB/s eta 0:00:02
   ---------------------------- ----------- 7.9/11.1 MB 3.5 MB/s eta 0:00:01
   ----------

In [3]:
import gurobipy as gp
from gurobipy import Model, GRB, quicksum
import numpy as np
import time

options = {
            "WLSACCESSID":"YOUR-ID",
            "WLSSECRET":"YOUR-ID",
            "LICENSEID": ""#YOUR-ID IN INTEGER SO DELETE THE ""
}

def bounded_beam_search(w, h, n):
    """
    Fungsi heuristik untuk menghasilkan solusi awal feasible.
    """
    solution = np.zeros((w, h), dtype=int)  # Inisialisasi solusi awal kosong
    blocks = np.random.permutation(n) + 1  # Mengacak nomor blok dari 1 hingga n
    
    idx = 0
    for j in range(w):
        for i in range(h-2):  # h-2 memastikan ruang untuk reshuffling
            if idx >= n:
                break
            solution[j, i] = blocks[idx]
            idx += 1

    # Visualisasi solusi awal

    print("Solusi awal (Bounded Beam Search - Randomized1):")
    print(solution.T)  # Tampilkan solusi awal sebagai array (transpose untuk visualisasi vertikal)
    return solution

def prepare_instance_set(set_type):
    """
    Fungsi untuk mempersiapkan parameter instance dari Set1 atau Set2.
    """
    if set_type == 'Set1':
        w_values = [3]
        h_values = range(5, 8)  # h = 5 hingga 12
        n = lambda w, h: w * (h - 2)
    elif set_type == 'Set2':
        w_values = [6, 7, 8, 9, 10]
        h_values = [3, 4, 5, 6]
        n = lambda w, h: ((w-1)*h, w*h)
    else:
        raise ValueError("Invalid set_type. Use 'Set1' or 'Set2'.")
    return w_values, h_values, n

def get_phi_sets(initial_solution, n, pi_values):
    """
    Calculate φi,j,t and φi,t sets
    """
    w = initial_solution.shape[0]
    phi_sets = {}
    phi_t_sets = {}
    
    for i in range(n):
        for t in range(pi_values[i] + 1, i + 1):
            # Calculate φi,t (blocks reshuffled before time t)
            phi_t = set()
            for i_prime in range(t, i):
                if pi_values[i_prime] < t:
                    phi_t.add(i_prime)
            phi_t_sets[(i, t)] = phi_t
            
            # Calculate φi,j,t for each stack
            for j in range(w):
                phi_ijt = set()
                for i_prime in range(t, i):
                    if (pi_values[i_prime] >= t and
                        get_stack(initial_solution, i_prime + 1) == j):
                        phi_ijt.add(i_prime)
                phi_sets[(i, j, t)] = phi_ijt
                
    return phi_sets, phi_t_sets

def get_pi_values(initial_solution, n):
    """
    Calculate πi values for each block i.
    πi represents the first time period block i is reshuffled.
    """
    pi_values = {}
    for i in range(n):
        block = i + 1
        pos = np.where(initial_solution == block)
        if len(pos[0]) > 0:
            stack, height = pos[0][0], pos[1][0]
            min_above = float('inf')
            for h in range(height + 1, initial_solution.shape[1]):
                if initial_solution[stack, h] != 0 and initial_solution[stack, h] < block:
                    min_above = min(min_above, initial_solution[stack, h])
            pi_values[i] = min_above if min_above != float('inf') else i
    return pi_values

def get_stack(initial_solution, block):
    """Helper function to get the initial stack of a block"""
    pos = np.where(initial_solution == block)
    return pos[0][0] if len(pos[0]) > 0 else -1


def branch_and_cut_rbrp(w, h, n, initial_solution):
    """
    Implementasi Branch-and-Cut untuk Restricted Block Relocation Problem.
    """
    env = gp.Env(params=options)
    # Formulate problem


    model = Model("RBRP", env=env )
    model.setParam('TimeLimit', 3600)  # Batas waktu 1 jam
    model.setParam('Threads', 1)       # Single-thread

    # Calculate necessary sets and values
    pi_values = get_pi_values(initial_solution, n)
    phi_sets, phi_t_sets = get_phi_sets(initial_solution, n, pi_values)

    # Variables
    x = model.addVars(n, w, n, vtype=GRB.BINARY, name="x") # constraint 11
    y = model.addVars(n, w, n, vtype=GRB.BINARY, name="y") # constraint 12

    # constraints (2)-(7)
    ## constraint 2
    for i in range(n):
        for t in range(i+1):
            model.addConstr(quicksum(x[i, j, t] for j in range(w)) == 1)
    
    ## constraint 3
    for j in range(w):
        for t in range(n):
            model.addConstr(quicksum(x[i, j, t] for i in range(n)) <= h)
    
    ## constraint 4-7
    for i in range(n):
        for j in range(w):
            for t in range(i):
                model.addConstr(y[i, j, t] >= x[i, j, t] - x[i, j, t+1])
                model.addConstr(y[i, j, t] <= 1 - x[i, j, t+1])
                model.addConstr(y[i, j, t] <= x[t, j, t])
                model.addConstr(y[i, j, t] <= x[i, j, t])

    # Initial configuration constraints (10)
    for i in range(n):
        s_i = get_stack(initial_solution, i + 1)
        if s_i >= 0:
            model.addConstr(x[i, s_i, 0] == 1)

    # Constraint (13): Block stays in initial position until πi
    for i in range(n):
        for t in range(pi_values[i]):
            s_i = get_stack(initial_solution, i + 1)
            model.addConstr(x[i, s_i, t] == 1)

    # Constraint (14): Block cannot stay in certain stacks after reshuffling
    for i in range(n):
        for i_prime in range(i):
            if pi_values[i] <= pi_values[i_prime] and pi_values[i_prime] == i_prime:
                s_i_prime = get_stack(initial_solution, i_prime + 1)
                model.addConstr(x[i, s_i_prime, i_prime + 1] == 0)

    # Constraint (15): Block movement restrictions based on φi,j,t sets
    for i in range(n):
        for j in range(w):
            for t in range(pi_values[i] + 1, i):
                phi_ijt = phi_sets.get((i, j, t), set())
                phi_it = phi_t_sets.get((i, t), set())
                min_phi = min(phi_ijt.union(phi_it), default=i)
                if t < min_phi:
                    model.addConstr(x[i, j, t] <= x[i, j, t+1])

    # Constraint (16): Block movement restrictions based on all φ sets
    for i in range(n):
        for j in range(w):
            for t in range(pi_values[i] + 1, i):
                phi_ijt = phi_sets.get((i, j, t), set())
                phi_it = phi_t_sets.get((i, t), set())
                all_phi = phi_it.union(*[phi_sets.get((i, k, t), set())
                                       for k in range(w) if k != j])
                min_phi = min(all_phi, default=i)
                if t < min_phi:
                    model.addConstr(x[i, j, t] == x[i, j, t+1])
    
    # Constraint (17): Handle first reshuffle conditions
    for i in range(n):
        s_i = get_stack(initial_solution, i + 1)
        pi_plus_one = pi_values[i] + 1
        for j in range(w):
            if j != s_i:  # Only for stacks different from initial stack
                phi_ijt = phi_sets.get((i, j, pi_plus_one), set())
                if phi_ijt:  # If the set is not empty
                    min_phi = min(phi_ijt)
                    phi_it = phi_t_sets.get((i, pi_plus_one), set())
                    smaller_blocks = [i_p for i_p in phi_it if i_p < min_phi]
                    model.addConstr(
                        x[i, j, pi_plus_one] <= 
                        y[i, j, min_phi] + 
                        quicksum(y[i, j, i_p] for i_p in smaller_blocks)
                    )
    # Constraint (18): Surrogate for constraints (8) and (9)
    for i in range(n):
        for t in range(pi_values[i] + 1, n):
            phi_t = phi_t_sets.get((i, t), set())
            for i_prime in phi_t:
                for j in range(w):
                    phi_ijt = phi_sets.get((i, j, t), set())
                    if phi_ijt and min(phi_ijt) > i_prime:
                        model.addConstr(
                            y[i, j, i_prime] + quicksum(y[i, j, i_p] for i_p in phi_t) >= 
                            x[i, j, t] + x[i_prime, j, t-1] - x[i, j, t-1] - 1,
                            name=f"Surrogate_{i}_{j}_{t}_{i_prime}"
                        )

    # Objective function
    model.setObjective(quicksum(y[i, j, t]
                               for i in range(n)
                               for j in range(w)
                               for t in range(i)),
                      GRB.MINIMIZE)
    
    # Start the timer
    start_time = time.time()
    model.setParam("OutputFlag", 1)

    model.optimize()
    end_time = time.time()

    optimization_time = end_time - start_time


    total_instances = sum(1 for _ in range(n))
    solved_instances = sum(1 for i in range(n) if model.status == GRB.OPTIMAL)
        
    # Output solution
    if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
        opt_status = solved_instances
        avg_time_solved = optimization_time
        avg_cuts = model.getAttr(GRB.Attr.NumConstrs)    
        avg_gap_final = 0  # Fully solved            
        print("Solution found:")
        for i in range(n):
            for j in range(w):
                for t in range(n):
                    if x[i, j, t].x > 0.5:
                        print(f"Block {i+1} in Stack {j+1} at Time {t+1}")
        # Inisialisasi matriks solusi per waktu
        solution_matrix = np.zeros((n, w, h), dtype=int)

        for i in range(n):
            for j in range(w):
                for t in range(n):
                    if x[i, j, t].x > 0.5:
                        # Temukan ketinggian pertama yang kosong di stack j pada waktu t
                        height = np.argmax(solution_matrix[t, j, :] == 0)
                        solution_matrix[t, j, height] = i + 1

        # Tampilkan solusi per waktu
        for t in range(n):
            print(f"Time {t + 1}:")
            print(solution_matrix[t].T)  # Transpose untuk visualisasi vertikal
                      
    else:
        print("No feasible solution found.")
        opt_status = f"{solved_instances}/{total_instances}" 
        avg_time_solved = '-'
        avg_cuts = '-'
        avg_gap_final = model.MIPGap * 100 if model.MIPGap is not None else '-'        

    # Calculate statistics
    stats = {
        'w': w,
        'h': h,
        'n': n,
        'I' : sum(1 for i in range(n)),
        'Opt': opt_status,        
        'Vars': model.numVars,
        'Cons': model.numConstrs,
        'Glp': model.ObjVal if model.status == GRB.OPTIMAL else 0.0,
        'Tlp': optimization_time,
        'Nodes': model.NodeCount, #model.BarIterCount
        'TC': optimization_time,  # Total computation time
        'T': avg_time_solved,
        'Cuts': avg_cuts,
        'Gf': avg_gap_final        
    }
    
    return stats

def run_experiment(w, h):
    n = w * (h - 2)  # Menentukan jumlah blok berdasarkan w dan h
    initial_solution = bounded_beam_search(w, h, n)
    stats = branch_and_cut_rbrp(w, h, n, initial_solution)
    return stats


# Menyiapkan instance dan menjalankan optimasi
w_values, h_values, n_function = prepare_instance_set("Set1")
for w in w_values:
    for h in h_values:
        n = n_function(w, h)
        initial_solution = bounded_beam_search(w, h, n)
        stats = branch_and_cut_rbrp(w, h, n, initial_solution)
        print(stats)

def run_multiple_experiments():
    set_type = 'Set1'
    w_values, h_values, _ = prepare_instance_set(set_type)
    instances = [(w, h) for w in w_values for h in h_values]

    results = []
    for w, h in instances:
        result = run_experiment(w, h)
        results.append(result)
    
    return results

# Collect and print results
results = run_multiple_experiments()
for result in results:
    print(result)        



Solusi awal (Bounded Beam Search - Randomized1):
[[5 4 1]
 [6 3 9]
 [7 2 8]
 [0 0 0]
 [0 0 0]]


TypeError: an integer is required